In [1]:
import numpy as np
import pandas as pd
import sqlite3
from statsmodels.stats.power import NormalIndPower
from statsmodels.stats.proportion import proportion_effectsize

# Loading data and getting actual rates
conn = sqlite3.connect('../data/hillstrom.db')
df = pd.read_sql("SELECT * FROM experiment", conn)

treatment = df[df['treatment'] == 'Womens E-Mail']
control = df[df['treatment'] == 'No E-Mail']

rate_treatment = treatment['visit'].mean()
rate_control = control['visit'].mean()
n_control = len(control)
lift = rate_treatment - rate_control

In [2]:

# ==========================================
# 4.1 — Practical Significance
# ==========================================
extra_visits_per_1000 = lift * 1000
cost_per_1000 = 1000 * 0.10  # Assuming $0.10/email cost

print("--- PRACTICAL SIGNIFICANCE ---")
print(f"Extra visits per 1,000 emails: {extra_visits_per_1000:.1f}")
print(f"Cost to send 1,000 emails:     ${cost_per_1000:.2f}")
print(f"Cost per incremental visit:    ${cost_per_1000 / extra_visits_per_1000:.2f}\n")

# ==========================================
# 4.2 — Retrospective Power
# ==========================================
effect_size = proportion_effectsize(rate_treatment, rate_control)
retro_power = NormalIndPower().power(
    effect_size=effect_size, 
    nobs1=n_control, 
    alpha=0.05, 
    ratio=1.0
)

print("--- RETROSPECTIVE POWER ---")
print(f"Observed absolute lift: {lift:.4f}")
print(f"Retrospective power to detect this lift: {retro_power:.2%}\n")

# ==========================================
# 4.3 — Prospective Power
# ==========================================
# Target half the observed lift for a more realistic future baseline
half_lift = lift / 2
smaller_effect_size = proportion_effectsize(rate_control + half_lift, rate_control)

req_n = NormalIndPower().solve_power(
    effect_size=smaller_effect_size, 
    alpha=0.05, 
    power=0.80, 
    ratio=1.0,
    alternative='two-sided'
)

print("--- PROSPECTIVE POWER ---")
print(f"Target lift (half observed): {half_lift:.4f}")
print(f"Required users per arm for future test: {int(np.ceil(req_n)):,}")

--- PRACTICAL SIGNIFICANCE ---
Extra visits per 1,000 emails: 45.2
Cost to send 1,000 emails:     $100.00
Cost per incremental visit:    $2.21

--- RETROSPECTIVE POWER ---
Observed absolute lift: 0.0452
Retrospective power to detect this lift: 100.00%

--- PROSPECTIVE POWER ---
Target lift (half observed): 0.0226
Required users per arm for future test: 3,175


The experiment's massively overpowered (100%) to detect the 4.52% absolute lift it actually found. The practical significance is clear: it costs $2.21 to acquire an incremental site visit via this campaign. For someone designing the next version of this test—assuming future lifts might be more modest (~2.26%) as the novelty wears off—they will need approximately 3,175 users per arm to reliably detect that smaller effect. Our current sample size of ~21,300 is more than sufficient.